# Capítulo 5: Aprendizaje por Refuerzo Profundo: Deep Q-Networks (DQN)

En este cuaderno implementamos un agente **Double Deep Q-Network (Double DQN)** en **PyTorch** con **Experience Replay** y actualización suave de red objetivo (**Polyak soft-update**).


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque

# ============================================================
# 1. Arquitectura de la Red Q Profunda
# ============================================================
class RedQOptiBot(nn.Module):
    def __init__(self, dim_estado=4, n_acciones=4):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(dim_estado, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, n_acciones)
        )

    def forward(self, x):
        return self.red(x)


# ============================================================
# 2. Buffer de Repetición de Experiencias (Experience Replay)
# ============================================================
class BufferReplay:
    def __init__(self, capacidad=50000):
        self.memoria = deque(maxlen=capacidad)

    def guardar(self, s, a, r, s_sig, terminado):
        self.memoria.append((s, a, r, s_sig, float(terminado)))

    def muestrear_lote(self, batch_size=32):
        lote = random.sample(self.memoria, batch_size)
        s, a, r, s_sig, d = zip(*lote)
        
        # Conversión eficiente a tensores de PyTorch
        return (torch.FloatTensor(np.array(s)),
                torch.LongTensor(a).unsqueeze(1),
                torch.FloatTensor(r).unsqueeze(1),
                torch.FloatTensor(np.array(s_sig)),
                torch.FloatTensor(d).unsqueeze(1))

    def __len__(self):
        return len(self.memoria)


# ============================================================
# 3. Agente Deep Q-Network (Double DQN)
# ============================================================
class AgenteDQN:
    def __init__(self, dim_estado=4, n_acciones=4, lr=1e-3, gamma=0.99, tau=0.005):
        self.n_acciones = n_acciones
        self.gamma = gamma
        self.tau = tau
        self.epsilon = 1.0
        self.eps_min = 0.02
        self.decaimiento = 0.995

        # Red Online y Red Target
        self.red_online = RedQOptiBot(dim_estado, n_acciones)
        self.red_target = RedQOptiBot(dim_estado, n_acciones)
        self.red_target.load_state_dict(self.red_online.state_dict())

        self.optimizador = optim.Adam(self.red_online.parameters(), lr=lr)
        self.criterio_perdida = nn.SmoothL1Loss()  # Pérdida de Huber

    def seleccionar_accion(self, estado):
        # Exploración vs Explotación (epsilon-greedy)
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_acciones)
        
        with torch.no_grad():
            s_tensor = torch.FloatTensor(estado).unsqueeze(0)
            valores_q = self.red_online(s_tensor)
            return int(valores_q.argmax(dim=1).item())

    def aprender_de_lote(self, buffer, batch_size=32):
        if len(buffer) < batch_size:
            return 0.0

        estados, acciones, recompensas, estados_sig, terminados = buffer.muestrear_lote(batch_size)

        with torch.no_grad():
            # Double DQN: selección con red online, evaluación con red target
            mejores_acciones = self.red_online(estados_sig).argmax(dim=1, keepdim=True)
            q_futuro = self.red_target(estados_sig).gather(1, mejores_acciones)
            objetivos = recompensas + self.gamma * (1.0 - terminados) * q_futuro

        # Predicción de la red online para las acciones efectivamente tomadas
        q_predichos = self.red_online(estados).gather(1, acciones)

        # Cálculo de pérdida y retropropagación
        perdida = self.criterio_perdida(q_predichos, objetivos)
        self.optimizador.zero_grad()
        perdida.backward()
        self.optimizador.step()

        # Actualización suave de la red target (Polyak soft-update)
        for p_t, p_o in zip(self.red_target.parameters(), self.red_online.parameters()):
            p_t.data.copy_(self.tau * p_o.data + (1.0 - self.tau) * p_t.data)

        # Reducción progresiva de epsilon
        self.epsilon = max(self.eps_min, self.epsilon * self.decaimiento)
        return perdida.item()

print("Módulos de Double DQN instanciados correctamente.")


## Simulación de Prueba con Entorno Continuo Simulado


In [ ]:
# Instanciamos el agente y buffer
agente = AgenteDQN(dim_estado=4, n_acciones=4)
buffer = BufferReplay(capacidad=10000)

# Simulamos interacciones sintéticas
for _ in range(500):
    s = np.random.randn(4)
    a = agente.seleccionar_accion(s)
    r = np.random.randn()
    s_sig = s + 0.1 * np.random.randn(4)
    terminado = random.choice([False, False, False, True])
    buffer.guardar(s, a, r, s_sig, terminado)
    agente.aprender_de_lote(buffer, batch_size=32)

print(f"Buffer con {len(buffer)} transiciones.")
print(f"Epsilon actual: {agente.epsilon:.4f}")
print("Entrenamiento funcional verificado exitosamente.")
